# LLM track

This notebook is the way to ask LLMs about NMR problems via OpenRouter.
Each molecule is shown to the model as its 1H and 13C NMR. A general-purpose LLM reads the two
spectra and returns up to ten ranked SMILES, best first.

This notebook does one thing end to end — query the models and write the raw answers. It builds the
prompt, calls each model through OpenRouter, records whether the reply parsed, and appends one JSON
line per call to `results/LLM_results/llm_final_raw.jsonl`. 


## 1 · Setup

In [1]:
from __future__ import annotations

import json
import re

from rdkit import Chem
from rdkit import RDLogger

RDLogger.DisableLog("rdApp.*")

In [2]:
import os
from pathlib import Path


def _find_repo_root(start=None):
    """Repo root = the folder that holds dataset/ and results/.

    Anchored on a real file rather than on the working directory, so the notebook works whether
    Jupyter was started in the repo root or in dataset/ (its own folder, the usual default).
    """
    p = (start or Path.cwd()).resolve()
    for d in (p, *p.parents):
        if (d / "dataset" / "dataset_selected_clean_105.json").exists():
            return d
    raise FileNotFoundError("run this notebook from inside the NMRArena repo")


REPO_ROOT = _find_repo_root()
DATA_PATH = REPO_ROOT / "dataset" / "dataset_selected_clean_105.json"
RUNS_PATH = REPO_ROOT / "results" / "LLM_results" / "llm_final_raw.jsonl"
RUNS_PATH.parent.mkdir(parents=True, exist_ok=True)

## 2 · Configuration


In [3]:
NUM_CANDIDATES = 10
MIN_SLOTS = 10

MODELS = {
    "claude":   "anthropic/claude-opus-4.8",      # Claude Opus 4.8
    "gpt":      "openai/gpt-5.5",                  # GPT-5.5
    "gemini":   "google/gemini-3.1-pro-preview",  # Gemini 3.1 Pro Preview
    "deepseek": "deepseek/deepseek-v4-pro",        # DeepSeek V4 Pro
    "qwen":     "qwen/qwen3.7-max",               # Qwen 3.7 Max
    "grok":     "x-ai/grok-4.3",                  # Grok 4.3
}


TEMPERATURE = 1.0
MAX_TOKENS  = 24576   # 24K

MODELS

{'claude': 'anthropic/claude-opus-4.8',
 'gpt': 'openai/gpt-5.5',
 'gemini': 'google/gemini-3.1-pro-preview',
 'deepseek': 'deepseek/deepseek-v4-pro',
 'qwen': 'qwen/qwen3.7-max',
 'grok': 'x-ai/grok-4.3'}

## 3 · Chemistry helpers


In [4]:
def canonical(smiles: str, strip_stereo: bool = True):
    s = (smiles or "").strip().strip("`").strip()
    if not s:
        return None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None
    return Chem.MolToSmiles(m, isomericSmiles=not strip_stereo)


def _normalize_nmr(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^\s*[\dHC]*_?NMR\s*", "", s)      # drop leading 1H/13C NMR label
    s = re.sub(r"_(d\d)", r"-\1", s)               # DMSO_d6 -> DMSO-d6
    s = re.sub(r"([A-Za-z])_(\d)", r"\1\2", s)     # CDCl_3 -> CDCl3
    s = re.sub(r"\s+", " ", s).strip()
    return s

## 4 · Prompt


In [5]:
_SYSTEM_TEMPLATE = r"""You are a senior organic chemist with decades of hands-on experience in NMR structure elucidation. 
You read 1D ¹H and ¹³C spectra the way an expert does: every chemical shift, multiplicity, J-coupling and integral is hard evidence that constrains the structure, and you reason from these patterns to the correct connectivity.

You will be given the experimental data as two lines:
- line 1: the ¹H NMR data (chemical shifts; usually with multiplicity, J-values in Hz and integration).
- line 2: the ¹³C NMR data (a list of chemical shifts with multiplicity if applicable).
No molecular formula or mass is provided — you must infer the constitution from the NMR data alone.

Systematic method (apply every time):
1. Parse each spectrum. List every ¹H signal (δ, multiplicity, J, integration) and every ¹³C signal (δ).
2. Proton count: sum the integrals for the (relative) number of H. An odd total often implies an odd number of N or of a monovalent heteroatom such as a halogen.
3. Carbon count & symmetry: number of ¹³C signals = number of chemically distinct carbons. If this is fewer than the carbons implied by the ¹H pattern (e.g. a 6H doublet = two equivalent CH₃ = one ¹³C line), invoke molecular symmetry/equivalence.
4. Read multiplicities as coupling information (n+1 rule): s = no coupled neighbours (isolated CH₃/CH₂, e.g. OCH₃, ArCH₃, or next to a quaternary/heteroatom); d = 1 neighbour; t = 2 equivalent neighbours; q = 3 (classic CH₂ of an ethyl); septet = 6 (isopropyl methine); dd/ddd = couplings to inequivalent protons; br/m = exchangeable or overlapping. Mutually coupled nuclei share the same J — use J-values to pair partners.
5. Classify each ¹³C shift by region (below) to assign functional groups.
6. Cross-check ¹H vs ¹³C: every proposed group must be consistent in BOTH spectra (e.g. an aldehyde needs ¹H ~9–10 AND ¹³C ~190–205).
7. Assemble fragments into a skeleton, build a working molecular formula, compute the degrees of unsaturation, and confirm that rings/π-systems match the aromatic/alkene/carbonyl evidence and that every observed signal is accounted for.

Diagnostic patterns you actively use —

Characteristic ¹H groupings:
- Ethyl: t (~1.0–1.3, 3H) + q (~2.3–4.2, 2H). Isopropyl: d (~0.9–1.3, 6H) + septet (1H). tert-Butyl: s (~0.9–1.4, 9H). Isobutyl/isoamyl: d (6H) + methine (m, 1H) + CH₂.
- OCH₃: s ~3.3–3.9 (¹³C ~52–56). N-CH₃: s ~2.2–3.0. Acetyl CH₃: s ~2.1.
- Aromatic: mono-substituted ≈ 5H near 7.2–7.4; para-disubstituted = AA′BB′ (two ~2H doublets); ortho/meta give distinct dd/td patterns. Alkene H ~4.5–6.5 with diagnostic ³J (cis ~6–12, trans ~12–18 Hz).
- Exchangeable OH/NH/COOH: broad, variable shift (OH ~1–5, COOH ~10–13), often without visible coupling.
- Aldehyde: ¹H 9–10.

¹³C shift regions:
- 195–220 ketone/aldehyde C=O; 165–185 acid/ester/amide C=O; 110–150 aromatic & alkene (quaternary/ipso aromatic ~125–150); 90–110 acetal/anomeric/alkyne; 60–90 C–O (alcohol/ether); 40–60 C–N and α-to-carbonyl CH; 14–45 aliphatic CH/CH₂/CH₃; 5–25 CH₃ and shielded carbons.

SMILES requirements (answers are graded by exact structure match, so be precise):
- Each candidate is ONE valid, covalently-bonded, net-neutral molecule. No salts, counterions or disconnected fragments (no '.'), no isotope labels, no R-groups/wildcards.
- Groups that genuinely require formal charges (nitro, azide, N-oxide, diazonium, ylides) use their standard charged SMILES.
- Ignore stereochemistry entirely (no @, no /\, no E/Z).
- Write clean, chemically sensible SMILES.

Candidate selection & ranking (scoring rewards having the correct structure anywhere in the list, ranked as high as possible):
- Rank 1 = the single structure most fully and consistently supported by all the data.
- Use the remaining slots for genuinely DISTINCT constitutional isomers that are also consistent with the data — vary positional or regiochemistry, ring size, chain branching, heteroatom placement — to cover the realistic possibilities so the true structure is captured.
- Provide up to {n} structures and fill at least {min_slots} slots. Never include a candidate that contradicts the spectra, and never two that are the same canonical structure.

Output your final answer as a SINGLE fenced JSON code block, with nothing after it, in exactly this shape, no markdown codeblock, no prose:

{{"candidates": [{{"rank": 1, "smiles": "..."}}, {{"rank": 2, "smiles": "..."}}]}}
"""

def system_prompt(n: int = NUM_CANDIDATES) -> str:
    return _SYSTEM_TEMPLATE.format(n=n, min_slots=min(MIN_SLOTS, n))

In [6]:
def build_user_prompt(record: dict, n: int = NUM_CANDIDATES) -> str:
    lines = ["Determine the structure of a single organic molecule from its NMR data."]
    lines.append(f"1H NMR: {_normalize_nmr(record.get('h_nmr', ''))}")
    lines.append(f"13C NMR: {_normalize_nmr(record.get('c_nmr', ''))}")
    lines.append(f"\nPropose up to {n} candidate structures, ranked best-first, in the required JSON format.")
    return "\n".join(lines)


def build_messages(record: dict, n: int = NUM_CANDIDATES) -> list:
    return [
        {"role": "system", "content": system_prompt(n)},
        {"role": "user", "content": build_user_prompt(record, n)},
    ]

## 5 · Response parsing


In [ ]:
def _clean_json(s: str) -> str:
    s = re.sub(r"/\*.*?\*/", "", s, flags=re.S)   # /* ... */ comments
    s = re.sub(r"//[^\n]*", "", s)                # // line comments
    s = re.sub(r",\s*([}\]])", r"\1", s)          # trailing commas before } or ]
    return s


def _try_load(s: str):
    for cand in (s, _clean_json(s)):
        try:
            return json.loads(cand)
        except Exception:
            pass
    return None


def _coerce_candidate_list(obj):
    """Parsed JSON -> list of raw SMILES strings (rank-ordered) or None."""
    items = None
    if isinstance(obj, dict):
        if isinstance(obj.get("candidates"), list):
            items = obj["candidates"]
        elif "smiles" in obj or "SMILES" in obj:
            items = [obj]
    elif isinstance(obj, list):
        items = obj
    if not items:
        return None

    def sort_key(pair):
        i, it = pair
        if isinstance(it, dict) and isinstance(it.get("rank"), (int, float)):
            return (0, it["rank"], i)
        return (1, 0, i)

    out = []
    for _, it in sorted(enumerate(items), key=sort_key):
        if isinstance(it, str):
            out.append(it)
        elif isinstance(it, dict):
            sm = it.get("smiles") or it.get("SMILES")
            if isinstance(sm, str):
                out.append(sm)
    return out or None


def _match_braces(text: str, start: int) -> int:
    """Index of the '}' that closes the '{' at `start`, or -1 if unbalanced."""
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                return i
    return -1


def _extract_json_candidates(text: str):

    idx = text.rfind('"candidates"')
    if idx == -1:
        return None
    ob = text.rfind("{", 0, idx)
    if ob == -1:
        return None
    cb = _match_braces(text, ob)
    if cb == -1:
        return None
    obj = _try_load(text[ob:cb + 1])
    if obj is None:
        return None
    return _coerce_candidate_list(obj)


def parse_candidates(text: str, n: int = NUM_CANDIDATES, strip_stereo: bool = True) -> dict:
    text = text or ""
    raw = _extract_json_candidates(text)
    if not raw:
        return {"status": "format_fail", "candidates": []}

    out, seen = [], set()
    for s in raw:
        c = canonical(s, strip_stereo=strip_stereo)
        if c is None or c in seen:
            continue
        seen.add(c)
        out.append(c)
        if len(out) >= n:
            break
    return {"status": "ok" if out else "format_fail", "candidates": out}

## 6 · OpenRouter client


In [ ]:
from openai import OpenAI

# The key is read from the environment. Never paste it into the notebook: this repo is public.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("set OPENROUTER_API_KEY in your environment before running")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
print("OpenRouter client ready")

## 7 · Model call



In [ ]:
import time

_TRANSIENT = ("rate", "timeout", "timed out", "429", "500", "502", "503", "504",
              "overloaded", "connection", "temporar", "unavailable")


def _is_transient(exc) -> bool:
    s = f"{type(exc).__name__} {exc}".lower()
    return any(t in s for t in _TRANSIENT)


def _usage_get(u, k):
    if u is None:
        return None
    return u.get(k) if isinstance(u, dict) else getattr(u, k, None)


def call_openrouter_meta(messages, model, temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                         max_retries=3, backoff=2.0):
    extra_body = {"usage": {"include": True}}   # OpenRouter -> usage.cost

    for attempt in range(1, max_retries + 1):
        t0 = time.perf_counter()
        try:
            r = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
                extra_headers={"X-Title": "NMR-Arena"},
                extra_body=extra_body,
            )
            dt = time.perf_counter() - t0
            msg = r.choices[0].message
            text = msg.content or getattr(msg, "reasoning", "") or ""
            u = r.usage
            return {
                "response_text": text,
                "reasoning_text": getattr(msg, "reasoning", "") or "",
                "finish_reason": r.choices[0].finish_reason,
                "model_served": getattr(r, "model", None),
                "usage": {
                    "prompt_tokens": _usage_get(u, "prompt_tokens"),
                    "completion_tokens": _usage_get(u, "completion_tokens"),
                    "total_tokens": _usage_get(u, "total_tokens"),
                    "cost": _usage_get(u, "cost"),
                },
                "latency_s": round(dt, 3),
                "attempts": attempt,
                "gen_id": getattr(r, "id", None),
                "status": "ok" if text.strip() else "empty",
                "error": None,
            }
        except Exception as exc:
            if attempt < max_retries and _is_transient(exc):
                time.sleep(backoff ** attempt)
                continue
            return {
                "response_text": "", "reasoning_text": "", "finish_reason": None,
                "model_served": None,
                "usage": {"prompt_tokens": None, "completion_tokens": None,
                          "total_tokens": None, "cost": None},
                "latency_s": round(time.perf_counter() - t0, 3),
                "attempts": attempt, "gen_id": None,
                "status": "api_error",
                "error": {"type": type(exc).__name__, "message": str(exc)},
            }

## 8 · Run and logging


In [ ]:
import uuid
import threading
import datetime

SCHEMA_VERSION = 1
_LOG_LOCK = threading.Lock()


def _now_iso():
    return datetime.datetime.now(datetime.timezone.utc).isoformat()


def _json_safe(o):
    if isinstance(o, float):
        return None if (o != o or o in (float("inf"), float("-inf"))) else o
    if isinstance(o, dict):
        return {k: _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_json_safe(v) for v in o]
    return o


_SOLVENT_RE = re.compile(r"(CDCl3|DMSO-?d6|CD3OD|CD2Cl2|C6D6|D2O|CD3CN|acetone-?d6)", re.I)


def _solvent_from_nmr(record):
    for key in ("h_nmr", "c_nmr"):
        m = _SOLVENT_RE.search(record.get(key, "") or "")
        if m:
            return m.group(1)
    return None


def run_and_log(record, model_key, n=NUM_CANDIDATES, primary_class=None, path=RUNS_PATH):
    if model_key not in MODELS:
        raise ValueError(f"unknown model_key {model_key!r}; one of {sorted(MODELS)}")
    model_id = MODELS[model_key]
    messages = build_messages(record, n)

    meta = call_openrouter_meta(messages, model_id)
    parsed = parse_candidates(meta["response_text"], n=n)

    if meta["status"] == "api_error":
        status = "api_error"
    elif meta["status"] == "empty":
        status = "empty"
    elif parsed["status"] == "format_fail":
        status = "format_fail"
    else:
        status = "ok"

    row = {
        "schema_version": SCHEMA_VERSION,
        "run_id": str(uuid.uuid4()),
        "ts": _now_iso(),
        "model_key": model_key,
        "model_id": model_id,
        "model_served": meta["model_served"],
        "compound_id": record.get("compound_id"),
        "primary_class": primary_class,
        "solvent": _solvent_from_nmr(record),
        "messages": messages,
        "prompt": messages[-1]["content"],
        "response_text": meta["response_text"],
        "reasoning_text": meta["reasoning_text"],
        "finish_reason": meta["finish_reason"],
        "usage": meta["usage"],
        "latency_s": meta["latency_s"],
        "attempts": meta["attempts"],
        "gen_id": meta["gen_id"],
        "status": status,
        "error": meta["error"],
        "truth": {"true_smiles": canonical(record.get("smiles", ""))},
    }
    row = _json_safe(row)
    with _LOG_LOCK:
        with open(path, "a", encoding="utf-8") as f:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
            os.fsync(f.fileno())
    return row

## 9 · Run: six models × 105 compounds

In [11]:
import concurrent.futures as cf
from collections import Counter

# --- run settings ------------------------------------------------------------
RUN_MODELS  = list(MODELS)                 # all six models
MAX_WORKERS = 10                           # concurrent OpenRouter calls
RETRY_STATUSES = {"api_error", "empty"}    # rerun these; ok / format_fail count as done
# -----------------------------------------------------------------------------


def load_records(path=DATA_PATH):
    """Flatten dataset_selected_clean_105.json ({class: {decile: record}}) ->
    list of (primary_class, record)."""
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    out = []
    for cls, grp in d.items():
        if isinstance(grp, dict):
            for rec in grp.values():
                if isinstance(rec, dict) and "smiles" in rec:
                    out.append((cls, rec))
    return out


def load_done(path=RUNS_PATH, retry_statuses=RETRY_STATUSES):
    """Set of already-logged (model_key, compound_id) keys, so the sweep is resumable.
    Rows whose status is in retry_statuses are not counted as done."""
    done = set()
    if not path.exists():
        return done
    for line in path.open(encoding="utf-8"):
        try:
            r = json.loads(line)
        except Exception:
            continue
        if r.get("status") in retry_statuses:
            continue
        done.add((r.get("model_key"), r.get("compound_id")))
    return done


def _complexity(rec):
    c = rec.get("n_complex")
    return c if isinstance(c, (int, float)) else 1.0


records = load_records()
records.sort(key=lambda cr: _complexity(cr[1]))          # easy -> hard

tasks = [(mk, cls, rec) for cls, rec in records for mk in RUN_MODELS]
done = load_done()
pending = [t for t in tasks if (t[0], t[2].get("compound_id")) not in done]

print(f"{len(records)} compounds x {len(RUN_MODELS)} models = {len(tasks)} calls "
      f"| done: {len(tasks) - len(pending)} | to run: {len(pending)}")


def _one(t):
    mk, cls, rec = t
    return run_and_log(rec, mk, primary_class=cls)


rows = []
with cf.ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for i, row in enumerate(ex.map(_one, pending), 1):
        rows.append(row)
        print(f"[{i}/{len(pending)}] {row['model_key']:9s} {str(row['compound_id'])[:8]} "
              f"{row['status']:11s} {row['latency_s']:.0f}s")

if rows:
    print(f"\nwrote {len(rows)} rows -> {RUNS_PATH}")
    print("status:", dict(Counter(r['status'] for r in rows)))
else:
    print("nothing to run — everything is already logged")